# Deep Research

### By: Bruno Conterato

## 0. Imports and Config


In [1]:
!pip install -q openai-agents pydantic

In [2]:
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    AsyncOpenAI,
    ModelSettings,
)
from pydantic import BaseModel
from pydantic.dataclasses import dataclass
import asyncio

In [3]:
@dataclass
class CONFIG:
    model: str = "qwen2.5:7b-instruct-q4_0"
    num_search_terms: int = 3
    debug_mode: bool = False

    query_example: str = """What are the best tools for developing multi-agents AI systems using Python?"""

## 1. Tools and Definitions


### 1.1. Model to run Ollama

In [4]:
ollama_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

model = OpenAIChatCompletionsModel(model=CONFIG.model, openai_client=ollama_client)

### 1.2. The tool for web search

In [5]:
import httpx
from trafilatura import fetch_url, extract


def extract_html_txt(body: str) -> str:
    return (
        extract(
            fetch_url(body),
            no_fallback=True,
            include_comments=False,
            include_links=False,
            include_formatting=False,
            include_images=False,
            include_tables=True,
            fast=True,
            favor_precision=True,
        )
        or ""
    )


@function_tool
async def search_web(query: str) -> str:
    """Asyncronously search the web based on the query."""

    async with httpx.AsyncClient() as client:
        try:
            response = await client.get(
                "http://localhost:4479/search/text",
                params={"query": query, "max_results": 5},
            )
            response.raise_for_status()
            results_arr = [
                f"Title: {r['title']}\nBody: {extract_html_txt(r['href'])}"
                for r in response.json()["results"]
            ]
            return "\n\n".join(results_arr)
        except Exception as e:
            return f"Search failed: {str(e)}"

## 2. The Agents Definitions

### 2.1. The search planner Agent

In [6]:
from pydantic import Field


class WebSearchItem(BaseModel):
    reason: str = Field(
        description="Your reasoning for why this search is important to the query."
    )

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="A list of web searches to perform to best answer the query."
    )


planner_agent = Agent(
    name="Planner",
    instructions=f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {CONFIG.num_search_terms} terms to query for.",
    model=model,
    output_type=WebSearchPlan,
)

In [7]:
# results = await Runner.run(planner_agent, CONFIG.query_example)

In [8]:
# results.final_output

### 2.2. The search Agent

In [9]:
search_agent = Agent(
    name="Search agent",
    instructions="You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself.",
    tools=[search_web],
    model=model,
    model_settings=ModelSettings(tool_choice="required"),
)

In [10]:
# results = await Runner.run(
#     search_agent, CONFIG.query_example
# )

In [11]:
# results.final_output

### 2.3. Writter Agent

In [12]:
class ReportData(BaseModel):
    short_summary: str = Field(
        description="A short 2-3 sentence summary of the findings."
    )

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(
        description="Suggested topics to research further"
    )


writer_agent = Agent(
    name="WriterAgent",
    instructions="You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words.",
    model=model,
    output_type=ReportData,
)

## 3. Agent Runner Functions


In [13]:
async def plan_searches(query: str) -> WebSearchPlan:
    if CONFIG.debug_mode:
        print("Planning searches...")
        print(f"User query: {query}")
    result = await Runner.run(planner_agent, f"Query: {query}")
    if CONFIG.debug_mode:
        print(f"Will perform searches: {result.final_output}")
    return result.final_output


async def process_search_query(item: WebSearchItem) -> str:
    search_result = await Runner.run(search_agent, item.query)
    return search_result.final_output


async def perform_searches(searchPlan: WebSearchPlan) -> str:
    if CONFIG.debug_mode:
        print("\nPerforming Searches...")
        print(f"Search plan: {searchPlan}")
    tasks = [asyncio.create_task(process_search_query(s)) for s in searchPlan.searches]
    results = await asyncio.gather(*tasks)
    response = ""
    for i, result in enumerate(results):
        response += f"Resultado da busca {i+1}:\n{result}"
    return response


async def write_report(info: str) -> ReportData:
    result = await Runner.run(writer_agent, info)
    return result.final_output

In [14]:
async def deep_research(query: str) -> ReportData:
    searchPlan: WebSearchPlan = await plan_searches(query)
    searchResults = await perform_searches(searchPlan)
    report = await write_report(searchResults)
    return report

In [15]:
result = await deep_research(CONFIG.query_example)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


In [16]:
from IPython.display import display, Markdown

display(Markdown(result.markdown_report))

# Evaluating Agentic AI Frameworks for Multi-Agent Systems

## Introduction

As artificial intelligence moves beyond individual tasks into more complex domains such as planning, reasoning, memory, tool use, and multi-step workflows, frameworks designed to support autonomous agent development are becoming increasingly critical. This report focuses on five leading Python frameworks that can be considered 'agentic,' meaning they enable the reliable execution of complex multi-agent systems.

The chosen frameworks—LangGraph, CrewAI, Autogen (Microsoft), LlamaIndex Agents, and GraphBit—are evaluated based on specific criteria such as tool execution, memory management, planning capabilities, error handling, state management, and workflow orchestration. The report aims to guide developers in selecting the most appropriate framework for their needs or in building a custom solution if necessary.

## What Makes a Framework Agentic?

An agentic AI framework must meet the following core requirements:
1. **Tool Execution**: Ability to execute APIs, code, databases, scrapers, file systems.
2. **Memory Management**: Supports both short-term context and long-term storage (like vector stores or retrieval-augmented generation).
3. **Planning Capabilities**: Tasks can be decomposed into reasoning steps.
4. **Error Handling Mechanisms**: Retry capabilities and validation of tools/checking constraints.
5. **State Management**: Control flow using graphs, state machines, or workflow-based orchestration.
6. **Workflow Orchestration**: Manage multistep tasks with branching logic.

Frameworks that do not meet these criteria are unlikely to be suitable for serious agent development.

## Evaluation of AGI Frameworks

### 1. LangGraph - The Best Python Framework for Complex Agent Flows

#### Strengths:
- Graph-based orchestration, ideal for deterministic workflows with DAG (Directed Acyclic Graph) support.
- Reentrancy and fault-tolerance features enhance robustness.
- Excellent ecosystem compatibility with other LangChain components.
- Suitable for complex, stateful pipelines that require branching logic.

#### Weaknesses:
- High complexity may deter users who need simpler solutions.
- Performance can be slower due to the Python-based workflow engine.

**Best For:**
Developers building applications with high state management and deterministic control flow requirements.

### 2. CrewAI - The Most Popular Multi-Agent Framework

#### Strengths:
- Clear role definitions for agents, simplifying team-based projects.
- Rapid prototyping capability due to simpler setup procedures.
- Easy integration of tools and utilities as part of multi-agent systems.

#### Weaknesses:
- Execution can be non-deterministic, leading to inconsistent behavior.
- Risk of infinite loops in conversational settings without careful management.

**Best For:**
Developers prioritizing ease-of-use for multi-agent setups or those looking for a more intuitive prototyping environment. Not recommended for production-grade systems requiring reliability and determinism.

### 3. Autogen (Microsoft) - Best Conversational Multi-Agent System

#### Strengths:
- Supports complex interactions through message passing between agents.
- Facilitates dialogue, negotiation, and coordination among autonomous entities.
- Suitable for human-in-the-loop scenarios ensuring quality over automation alone.

#### Weaknesses:
- Non-deterministic execution can lead to inconsistent agent behavior across runs.
- Risk of complex conversations falling into infinite cycles without proper oversight.

**Best For:**
Research labs, universities, and conversational AI systems where exploration is key but production reliability might not be the highest priority. Ideal for studying multi-agent interactions in controlled environments with human supervision.

### 4. LlamaIndex Agents - Best for Retrieval-based Multi-Agent Systems

#### Strengths:
- World-class retrieval capabilities through advanced indexing techniques.
- Robust mechanisms to store and utilize context across agent interactions.
- Easy integration of a wide range of tools into multi-agent workflows.

#### Weaknesses:
- Focus on retrieval and indexing means limited support for full workflow orchestration beyond basic sequence steps.
- Less suited for complex, multi-agent collaboration scenarios where branching logic is necessary.

**Best For:**
Researchers and developers building analytical agents or document-intensive workflows that involve heavy data retrieval and management. Less suitable for production environments requiring comprehensive agent coordination.

### 5. GraphBit - The Most Promising Production-Grade Agentic AI Framework

#### Strengths:
- High performance, leveraging Rust for deterministic workflows.
- Predictability in memory management, ensuring consistent behavior over time.
- Enterprise-level security and reliability features essential for robust production systems.
- Structured state and planning through advanced workflow engines.

#### Weaknesses:
- Newer ecosystem with fewer mature tools or integrations compared to established frameworks like LangGraph.
- Primarily focused on engineering tasks, potentially lacking in broader 

OPENAI_API_KEY is not set, skipping trace export
